# Задания к занятию №4

## Задание 1.1: Таинственный DataFrame

Дан код:
```
import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print("Original ID:", id(df))

def clean_data(data):
    data = data[data['A'] > 1]  # фильтруем строки
    print("Inside function ID:", id(data))
    data.loc[:, 'C'] = data['A'] * 2  # добавляем колонку
    return data

new_df = clean_data(df)
print("New df ID:", id(new_df))
print("Original df after function:")
print(df)
print("New df:")
print(new_df)

```

Вопросы:

1) Сколько различных объектов DataFrame существует в памяти после выполнения кода?
2) Почему df остался неизменным, если DataFrame $-$ mutable объект?
3) Что произойдёт, если внутри функции всместо
   ```
   data = data[data['A'] > 1]
   ```
   
   написать:
   
   ```
   data.drop(index=data[data['A'] <= 1].index, inplace=True)?
   ```

In [1]:
import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print("Original ID:", id(df))

def clean_data(data):
    data = data[data['A'] > 1]  # фильтруем строки
    print("Inside function ID:", id(data))
    data.loc[:, 'C'] = data['A'] * 2  # добавляем колонку
    return data

new_df = clean_data(df)
print("New df ID:", id(new_df))
print("Original df after function:")
print(df)
print("New df:")
print(new_df)

Original ID: 136818506403472
Inside function ID: 136818735031520
New df ID: 136818735031520
Original df after function:
   A  B
0  1  4
1  2  5
2  3  6
New df:
   A  B  C
1  2  5  4
2  3  6  6


/tmp/ipykernel_271/2200550542.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:, 'C'] = data['A'] * 2  # добавляем колонку


---

### 1 вопрос - 2 объекта (df и new_df старый и новый объект)


### 2 вопрос - мы никак не редактировали df и не изменяли (создали копию?)


### 3 вопрос - мы применили к нашей таблице изменения ```inplace=True``` то есть радактируем нашу основу, а не создаем копию (будет 1 объект)

---



## Задание 1.2: Списковое недоразумение

Объясните, почему следующий код работае не так, как мог бы ожидать новичок?

```
def add_and_multiply(numbers, multiplier=[]):
    multiplier.append(1)
    return [n * len(multiplier) for n in numbers]

print(add_and_multiply([1, 2, 3]))  # Ожидание: [1, 2, 3]?
print(add_and_multiply([1, 2, 3]))  # Ожидание снова [1, 2, 3]? А получаем...
print(add_and_multiply([1, 2, 3], [5, 5]))  # А тут?
```

Исправьте функцию так, чтобы она работала предсказуемо, но сохранила возможность передавать свой список `multiplier`.

In [ ]:
def add_and_multiply(numbers, multiplier=[]):

    multiplier = multiplier.copy() # можно добавить копирование чтобы избежать влияния на следующие вызовы

    multiplier.append(1)
    return [n * len(multiplier) for n in numbers]

print(add_and_multiply([1, 2, 3]))
print(add_and_multiply([1, 2, 3]))
print(add_and_multiply([1, 2, 3], [5, 5]))

[1, 2, 3]
[1, 2, 3]
[3, 6, 9]


---


## Задание 2.1: Собственный OneHotEncoder

Создайте класс `SimpleOneHotEncoder`, который будет обучаться на списке категорий и преобразовывать новые данные.

Требования:
- Метод `fit(categories)` принимает список строк или `pandas.Series`;
- Метод `transform(categories)` возвращает pandas.DataFrame с one-hot закодированными колонками (порядок колонок должен соответствовать порядку уникальных категорий из `fit`);
- Метод `fit_transform(categories)` $-$ выполняет обе операции и возвращает результат.

**Важно**: Если на этапе `transform` встречается категория, которой не было в `fit`, метод должен выбросить понятное исключение `ValueError` с указанием "неизвестной категории".

#### Подсказки:
- храните уникальные значения в атрибуте `self.categories_`;
- подумайте о том, что произойдёт с памятью, если передать огромный список в `fit`;
- что такое one-hot encoding и зачем он нужен: <a href=https://sky.pro/wiki/analytics/one-hot-encoding-chto-eto-takoe-i-kak-primenyat-v-mashinnom-obuchenii/>тык</a>;
- про модель исключений в python: <a href=https://education.yandex.ru/handbook/python/article/model-isklyuchenij-python-try-except-else-finally-moduli>тык</a>.

In [ ]:
import pandas as pd
import numpy as np

arr = pd.Series(['blue', 'green', 'red', 'red'])

class SimpleOneHotEncoder:
    def __init__(self):
        self.categories_ = None


    def fit(self, data, sorting=True):

        self.categories_ = np.unique(data)

        if sorting:
            self.categories_.sort()
        return self


    def transform(self, data):
        if self.categories_ is None:
            raise ValueError("неизвестной категории")

        count_categ = len(self.categories_)
        count_data = len(data)

        matrix = np.zeros((count_data, count_categ))

        df = pd.DataFrame(matrix, columns=self.categories_)

        for i, j in enumerate(data):
            if j in self.categories_:
                df.loc[i, j] = 1

        return df


    def fit_transform(self, data, sorting=True):
        return self.fit(data, sorting).transform(data)



In [ ]:
OHE = SimpleOneHotEncoder()

OHE.fit(arr)
data = OHE.transform(arr)
data

,blue,green,red
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0


In [ ]:
OHE_W = SimpleOneHotEncoder()

data_W = OHE_W.fit_transform(arr)
data_W

,blue,green,red
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0


## Задание 3.1: Кэширующий трансформер

Представьте, что у вас есть "тяжёлая" функция предобработки текста (очистка, стемминг/лемматизация). Создайте класс `CachedTransformer`, который запоминает результаты обработки для повторяющихся входов.

In [6]:
class CachedTransformer:
    def __init__(self, transformer_func):
        self.transformer_func = transformer_func
        self.cache = {}  # хранилище: вход -> результат


    def transform(self, X):

        out = []

        for word in X:
            if self.cache[word] is not None:
                out.append(self.cache[word])
            else:
                new = self.transformer_func(word)
                self.cache[word] = new
                out.append(self.cache[word])

        return out


**Вопросы**:
- Какие проблемы могут возникнуть с памятью, если кэшировать абсолютно все уникальные строки?
- Как можно ограничить размер кэша (например, хранить только 1 000 последних уникальных значений)? (ответ на этот вопрос реализовать в коде).

## Финальный проект: мини-проект "свой sklearn"

Создайте класс `DataFeatureExtractor`, который из колонки с датами генерирует набор числовых признаков.

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

In [8]:
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, date_column, features=['year', 'month', 'day', 'weekday', 'month_sin', 'month_cos']):

        self.date_column = date_column
        self.features = features
        self.fitted_ = False

        self.min_date = None
        self.max_date = None

        self.name = None

    def fit(self, X, y=None):

        if self.date_column not in X.columns:
            raise ValueError("колонка отсутствует")
        
        if len([X.columns[0]]) != 1:

            raise ValueError("нужна одна колонка")

        else:
            date = pd.to_datetime(X[self.date_column], errors='coerce')

            if date.isna().all():
                raise ValueError("колонка не содержит дат")

            #так и не понял зачем
            self.max_date = max(date)
            self.min_date = min(date)

            self.fitted_ = True
            return self


    def transform(self, X):

        if self.date_column not in X.columns:
            raise ValueError("колонка не найдена")
        if not self.fitted_: #когда говорили с Али он сказал что забыл добавить проверку)))) 
            raise ValueError("вызовите fit")


        date = pd.to_datetime(X[self.date_column], errors='coerce')
        main = pd.DataFrame(index=X.index)


        for feature in self.features:
            if feature == 'year':
                main['year'] = date.dt.year

            elif feature == 'month':
                main['month'] = date.dt.month

            elif feature == 'day':
                main['day'] = date.dt.day

            elif feature == 'weekday':
                main['weekday'] = date.dt.weekday + 1


            #реализованно криво, мне не нравиться
            elif feature == 'month_sin':
                main['month_sin'] = np.sin(2 * np.pi * date.dt.month / 12)

            elif feature == 'month_cos':
                main['month_cos'] = np.cos(2 * np.pi * date.dt.month / 12)
            
            else:
                raise ValueError('неизвестный признак')

        self.name = list(main.columns)
        
        return main

    def get_feature_names_out(self, input_features=None):
        if self.name is None:
            raise ValueError('вызавите сначало transform')

        return self.name.copy()


In [10]:
# тестовые данные сгенерировал ИИ так как было разрешение


import pandas as pd
import numpy as np

# 1. Обычные даты
df1 = pd.DataFrame({
    'date': ['2022-01-15', '2023-02-20', '2023-03-25', '2023-04-10', '2023-05-05']
})

# 2. Даты с пропусками
df2 = pd.DataFrame({
    'date': ['2023-01-15', None, '2023-03-25', np.nan, '2023-05-05']
})

# 3. Разные названия колонок
df3 = pd.DataFrame({
    'dt': ['2023-01-15', '2023-02-20', '2023-03-25'],
    'other_column': [1, 2, 3]
})

# 4. Повторяющиеся даты
df4 = pd.DataFrame({
    'date': ['2023-01-15', '2023-01-15', '2023-02-20', '2023-02-20']
})

# 5. Одна дата
df5 = pd.DataFrame({
    'date': ['2023-06-15']
})

# 6. Пустой DataFrame
df6 = pd.DataFrame(columns=['date'])

# 7. Только пропуски
df7 = pd.DataFrame({
    'date': [None, np.nan, None]
})

# 8. Разные дни недели
df8 = pd.DataFrame({
    'date': ['2023-01-16', '2023-01-17', '2023-01-18', '2023-01-21', '2023-01-22']
})

# 9. Граничные даты
df9 = pd.DataFrame({
    'date': ['2023-01-01', '2023-12-31', '2024-02-29']
})

# 10. Больше данных (20 строк)
df10 = pd.DataFrame({
    'date': [
        '2023-01-01', '2023-01-05', '2023-01-10', '2023-01-15', '2023-01-20',
        '2023-02-01', '2023-02-05', '2023-02-10', '2023-02-15', '2023-02-20',
        '2023-03-01', '2023-03-05', '2023-03-10', '2023-03-15', '2023-03-20',
        None, '2023-04-05', '2023-04-10', np.nan, '2023-04-20'
    ]
})

In [45]:
a = DateFeatureExtractor(date_column='date')
a.fit(df1)
ad = a.transform(df1)
ad

,year,month,day,weekday,month_sin,month_cos
0,2022,1,15,6,0.500000,8.660254e-01
1,2023,2,20,1,0.866025,5.000000e-01
2,2023,3,25,6,1.000000,6.123234e-17
3,2023,4,10,1,0.866025,-5.000000e-01
4,2023,5,5,5,0.500000,-8.660254e-01


In [46]:
a.get_feature_names_out()

['year', 'month', 'day', 'weekday', 'month_sin', 'month_cos']

In [12]:
r = DateFeatureExtractor(date_column='date', features=['year', 'month', 'month_sin'])

r.fit(df4)
tet = r.transform(df4)
tet

,year,month,month_sin
0,2023,1,0.500000
1,2023,1,0.500000
2,2023,2,0.866025
3,2023,2,0.866025


In [14]:
tet1 = r.fit_transform(df4)
tet1

,year,month,month_sin
0,2023,1,0.500000
1,2023,1,0.500000
2,2023,2,0.866025
3,2023,2,0.866025


In [15]:
r.get_feature_names_out()

['year', 'month', 'month_sin']

In [16]:
par = r.get_params()
par

{'date_column': 'date', 'features': ['year', 'month', 'month_sin']}

In [17]:
r.set_params(features=['year', 'weekday'])
r.fit(df4)

,date_column,'date'
,features,"['year', 'weekday']"


In [19]:
r.transform(df4).columns

Index(['year', 'weekday'], dtype='object')